In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ac/MRREL.RRF.ac
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ae/MRREL.RRF.ae
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ad/MRREL.RRF.ad
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRCONSO.RRF.aa/MRCONSO.RRF.aa
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ab/MRREL.RRF.ab
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRSTY.RRF/MRSTY.RRF
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.aa/MRREL.RRF.aa
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRCONSO.RRF.ab/MRCONSO.RRF.ab
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRCONSO.RRF.ac/MRCONSO.RRF.ac
/kaggle/input/datasets/konicarokeya/mesh-complete/retrieval_corpus_MESH_COMPLETE.parquet


**# **Cell 0 — Concat RRF parts****

In [2]:
from pathlib import Path
import shutil

UMLS_BASE = Path('/kaggle/input/datasets/konicarokeya/umls-2025ab-parts')
UMLS_OUT  = Path('/kaggle/working/umls')
UMLS_OUT.mkdir(exist_ok=True)

def concat_parts(part_names, output_name):
    out_path = UMLS_OUT / output_name
    if out_path.exists():
        print(f'{output_name} already exists ({out_path.stat().st_size/1024**2:.0f} MB) — skipping')
        return out_path
    print(f'Building {output_name} from {len(part_names)} parts...')
    with open(out_path, 'wb') as out_f:
        for name in sorted(part_names):
            p = UMLS_BASE / name / name
            print(f'  + {name}  ({p.stat().st_size/1024**2:.0f} MB)')
            with open(p, 'rb') as in_f:
                shutil.copyfileobj(in_f, out_f)
    print(f'  done -> {out_path.stat().st_size/1024**2:.0f} MB total\n')
    return out_path

MRCONSO_PATH = concat_parts(
    ['MRCONSO.RRF.aa', 'MRCONSO.RRF.ab', 'MRCONSO.RRF.ac'],
    'MRCONSO.RRF'
)
MRREL_PATH = concat_parts(
    ['MRREL.RRF.aa', 'MRREL.RRF.ab', 'MRREL.RRF.ac',
     'MRREL.RRF.ad', 'MRREL.RRF.ae'],
    'MRREL.RRF'
)
MRSTY_PATH  = UMLS_BASE / 'MRSTY.RRF' / 'MRSTY.RRF'
CORPUS_PATH = Path('/kaggle/input/datasets/konicarokeya/mesh-complete/retrieval_corpus_MESH_COMPLETE.parquet')

print('='*50)
print(f'MRCONSO : {MRCONSO_PATH}')
print(f'MRREL   : {MRREL_PATH}')
print(f'MRSTY   : {MRSTY_PATH}  exists={MRSTY_PATH.exists()}')
print(f'CORPUS  : {CORPUS_PATH}  exists={CORPUS_PATH.exists()}')

Building MRCONSO.RRF from 3 parts...
  + MRCONSO.RRF.aa  (1024 MB)
  + MRCONSO.RRF.ab  (1024 MB)
  + MRCONSO.RRF.ac  (97 MB)
  done -> 2145 MB total

Building MRREL.RRF from 5 parts...
  + MRREL.RRF.aa  (1024 MB)
  + MRREL.RRF.ab  (1024 MB)
  + MRREL.RRF.ac  (1024 MB)
  + MRREL.RRF.ad  (1024 MB)
  + MRREL.RRF.ae  (89 MB)
  done -> 4185 MB total

MRCONSO : /kaggle/working/umls/MRCONSO.RRF
MRREL   : /kaggle/working/umls/MRREL.RRF
MRSTY   : /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRSTY.RRF/MRSTY.RRF  exists=True
CORPUS  : /kaggle/input/datasets/konicarokeya/mesh-complete/retrieval_corpus_MESH_COMPLETE.parquet  exists=True


**Cell 1 — Imports**

In [3]:
import gc
import math
import pickle
import shutil
from pathlib import Path
from collections import defaultdict
from itertools import combinations
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('Imports OK')

Imports OK


**Cell 2 — Config**

In [4]:
# ──  CONFIG ──────────────────────────────────────────────────────────

HKG_OUT = Path('/kaggle/working/hkg.pkl')   

TARGET_SABS = {
    'MSH',
    'SNOMEDCT_US',
    'RXNORM',
    'NCI',
    'ICD10CM',
}

SYNONYM_ONLY_SABS = {
    'LNC',
    'HPO',
    'GO',
    'OMIM',
    'MEDDRA',
    'MEDLINEPLUS',
    'ENTREZGENE',
    'HGNC',
}

MIN_COOCCUR = 5

ALLOWED_TUIS = {
    'T047','T048','T049','T050','T191','T046','T184',
    'T121','T109','T195','T200','T116','T126','T127',
    'T131','T125','T129','T130','T123','T122','T197',
    'T103','T120',
    'T023','T024','T025','T026','T029','T030','T031','T022',
    'T039','T040','T041','T042','T043','T044','T045','T038',
    'T061','T058','T059','T060','T065','T063','T062',
    'T016','T096','T098','T099','T032','T100','T080',
    'T008','T015','T007','T004','T005','T001','T002',
    'T011','T012','T013','T014',
    'T028','T086','T114','T087','T085','T088',
    'T033','T034','T201','T190',
    'T081','T077','T169','T090','T052','T079','T078',
    'T170','T171',
    'T074','T075','T073',
    'T037','T020','T019','T018','T017',
    'T168',
    'T053','T054','T055',
    'T069','T070',
    'T192',  # Receptor — binding sites, receptor types
    'T082',  # Spatial Concept — protein conformation, structural terms
    'T102',  # Group Attribute — sex factors, age group modifiers
    'T091',  # Biomedical Occupation or Discipline — family practice, electrophysiology
    'T196',  # Element, Ion, or Isotope — tritium, radiolabels, isotopes
}

print(f'Config OK')
print(f'  Node-creating SABs : {TARGET_SABS}')
print(f'  Synonym-only SABs  : {SYNONYM_ONLY_SABS}')
print(f'  MIN_COOCCUR        : {MIN_COOCCUR}')
print(f'  Semantic types     : {len(ALLOWED_TUIS)}')

Config OK
  Node-creating SABs : {'NCI', 'SNOMEDCT_US', 'MSH', 'RXNORM', 'ICD10CM'}
  Synonym-only SABs  : {'MEDLINEPLUS', 'LNC', 'MEDDRA', 'OMIM', 'HGNC', 'GO', 'HPO', 'ENTREZGENE'}
  MIN_COOCCUR        : 5
  Semantic types     : 102


**Cell 3 — Load corpus**

In [5]:
print('Loading corpus...')
df = pd.read_parquet(CORPUS_PATH)
print(f'Rows    : {len(df):,}')
print(f'Sources : {df["source"].value_counts().to_dict()}')

def normalize_mesh(m):
    if m is None: return []
    if isinstance(m, np.ndarray): m = m.tolist()
    if not isinstance(m, list): return []
    out = []
    for x in m:
        if isinstance(x, dict):
            term   = x.get('term', '')
            meshid = x.get('mesh_id', '')
            if term and str(term).strip():
                raw_id = str(meshid).strip() if meshid else ''
                # Guard against literal string 'None' from Parquet serialization
                clean_id = raw_id if (raw_id and raw_id != 'None') else ''
                out.append({
                    'term'   : str(term).strip().lower(),
                    'mesh_id': clean_id
                })
    return out

df['meshes_norm'] = df['meshes'].apply(normalize_mesh)

term_to_chunks   = defaultdict(list)
meshid_to_chunks = defaultdict(list)

for idx, mesh_list in enumerate(tqdm(df['meshes_norm'], desc='Building chunk index')):
    for entry in mesh_list:
        t = entry['term']
        m = entry['mesh_id']
        if t:
            term_to_chunks[t].append(idx)
        if m and m.startswith('D'):
            meshid_to_chunks[m].append(idx)

corpus_texts = df['text'].tolist()

print(f'\nUnique terms    : {len(term_to_chunks):,}')
print(f'Unique mesh_ids : {len(meshid_to_chunks):,}')
print(f'Corpus size     : {len(corpus_texts):,}')
gc.collect()

Loading corpus...
Rows    : 2,089,296
Sources : {'medrag_pubmed': 1000000, 'pqaa': 649412, 'medrag_wikipedia': 239348, 'pqau': 200536}


Building chunk index:   0%|          | 0/2089296 [00:00<?, ?it/s]


Unique terms    : 26,190
Unique mesh_ids : 26,177
Corpus size     : 2,089,296


20

**Cell 4 — Load MRSTY**

In [6]:
print('Loading MRSTY...')

allowed_cuis = set()
cui_to_tui   = {}  # stores FIRST allowed TUI seen, or last TUI if none allowed

with open(MRSTY_PATH, encoding='utf-8') as f:
    for line in tqdm(f, desc='MRSTY'):
        row = line.rstrip('\n').split('|')
        if len(row) < 2: continue
        cui = row[0]
        tui = row[1]
        # Keep first TUI seen per CUI (avoids silent overwrites)
        if cui not in cui_to_tui:
            cui_to_tui[cui] = tui
        if tui in ALLOWED_TUIS:
            allowed_cuis.add(cui)
            # Prefer an allowed TUI for display when multiple exist
            cui_to_tui[cui] = tui

print(f'Total CUIs       : {len(cui_to_tui):,}')
print(f'Allowed CUIs     : {len(allowed_cuis):,}')
gc.collect()

Loading MRSTY...


MRSTY: 0it [00:00, ?it/s]

Total CUIs       : 3,488,973
Allowed CUIs     : 3,108,954


18

**Cell 5 — Load MRCONSO (fixed ispref bug)**

In [7]:
print('Loading MRCONSO...')

G             = nx.DiGraph()
label_to_cui  = {}
meshid_to_cui = {}
ispref_labels = set()  # track which labels were set by a preferred term

with open(MRCONSO_PATH, encoding='utf-8') as f:
    for line in tqdm(f, desc='MRCONSO'):
        row = line.rstrip('\n').split('|')
        if len(row) < 15: continue
        cui    = row[0]
        lang   = row[1]
        ispref = row[6]
        sab    = row[11]
        code   = row[13]
        name   = row[14]

        if lang != 'ENG': continue
        if sab not in TARGET_SABS: continue
        if cui not in allowed_cuis: continue

        label = name.lower().strip()

        if cui not in G:
            G.add_node(cui, label=label, sab=sab, chunk_idxs=[])

        if ispref == 'Y':
            G.nodes[cui]['label'] = label

        if label:
            # Preferred terms win over non-preferred on label collision
            if ispref == 'Y' or label not in ispref_labels:
                label_to_cui[label] = cui
                if ispref == 'Y':
                    ispref_labels.add(label)

        if sab == 'MSH' and code.startswith('D'):
            meshid_to_cui[code] = cui

print(f'Nodes loaded  : {G.number_of_nodes():,}')
print(f'label_to_cui  : {len(label_to_cui):,}')
print(f'meshid_to_cui : {len(meshid_to_cui):,}')
gc.collect()

Loading MRCONSO...


MRCONSO: 0it [00:00, ?it/s]

Nodes loaded  : 1,294,438
label_to_cui  : 3,275,438
meshid_to_cui : 29,121


18

In [8]:
# ── MRCONSO pass 2: synonym lookup table ───────────────────────────

import re

def _normalise(s):
    s = s.lower().strip()
    s = re.sub(r'\s*\(.*?\)', '', s)
    s = re.sub(r'[^a-z0-9\s]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

print('Building synonym lookup table (MRCONSO pass 2)...')

SAFE_LOOKUP_SABS = TARGET_SABS | SYNONYM_ONLY_SABS

# Raised from 3_000_000 — RunPod has no RAM issue, load everything
MAX_SYNONYMS = 10_000_000

synonym_to_cui  = {}
rxnorm_to_cui   = {}
syn_ispref_set  = set()
syn_count       = 0

with open(MRCONSO_PATH, encoding='utf-8') as f:
    for line in tqdm(f, desc='MRCONSO pass 2'):
        if syn_count >= MAX_SYNONYMS:
            break
        row = line.rstrip('\n').split('|')
        if len(row) < 15: continue
        cui    = row[0]
        lang   = row[1]
        ispref = row[6]
        sab    = row[11]
        code   = row[13]
        name   = row[14]

        if lang != 'ENG': continue
        if sab not in SAFE_LOOKUP_SABS: continue
        if cui not in G: continue

        label = name.lower().strip()
        if not label: continue

        if ispref == 'Y' or label not in syn_ispref_set:
            synonym_to_cui[label] = cui
            if ispref == 'Y':
                syn_ispref_set.add(label)
            syn_count += 1

        if sab == 'RXNORM' and code.isdigit():
            rxnorm_to_cui[code] = cui

del syn_ispref_set
gc.collect()

# Build normalised lookup — delta only (skip what label_to_cui already covers)
print('Building normalised index (delta only)...')
norm_to_cui = {}
for term, cui in tqdm(synonym_to_cui.items(), desc='Normalising'):
    if term in label_to_cui:
        continue
    n = _normalise(term)
    if n and n not in norm_to_cui and n not in label_to_cui:
        norm_to_cui[n] = cui

print(f'\nsynonym_to_cui  : {len(synonym_to_cui):,}')
print(f'rxnorm_to_cui   : {len(rxnorm_to_cui):,}')
print(f'norm_to_cui     : {len(norm_to_cui):,}')

# Cap warning — MRCONSO is sorted by CUI not SAB.
# If cap is hit, later SABs (HGNC, ENTREZGENE) may be silently missing.
print(f'\nEntries counted : {syn_count:,}  |  Cap : {MAX_SYNONYMS:,}')
if syn_count >= MAX_SYNONYMS:
    print('WARNING: cap was hit — some SABs near end of file may be missing.')
else:
    print('Cap not hit — all SAB entries loaded successfully.')

gc.collect()

Building synonym lookup table (MRCONSO pass 2)...


MRCONSO pass 2: 0it [00:00, ?it/s]

Building normalised index (delta only)...


Normalising:   0%|          | 0/3324271 [00:00<?, ?it/s]


synonym_to_cui  : 3,324,271
rxnorm_to_cui   : 224,477
norm_to_cui     : 45,318

Entries counted : 3,874,642  |  Cap : 10,000,000
Cap not hit — all SAB entries loaded successfully.


21

**Cell 6 — Load MRREL**

In [9]:
print('Loading MRREL...')

VALID_RELS = {'PAR', 'CHD', 'RO'}
edge_count = 0

with open(MRREL_PATH, encoding='utf-8') as f:
    for line in tqdm(f, desc='MRREL'):
        row = line.rstrip('\n').split('|')
        if len(row) < 8: continue
        cui1 = row[0]
        rel  = row[3]
        cui2 = row[4]
        rela = row[7]

        if rel not in VALID_RELS: continue
        if cui1 not in G: continue
        if cui2 not in G: continue

        if G.has_edge(cui1, cui2):
            existing = G[cui1][cui2]
            if rel not in existing['rel']:
                existing['rel']  += f'|{rel}'
                existing['rela'] += f'|{rela or rel}'
        else:
            G.add_edge(cui1, cui2, rel=rel, rela=rela or rel)

        edge_count += 1

print(f'UMLS rel entries processed : {edge_count:,}')
print(f'Unique directed edges      : {G.number_of_edges():,}')
print(f'Edges with multiple rels   : {sum(1 for u,v,d in G.edges(data=True) if "|" in str(d.get("rel",""))):,}')
gc.collect()

Loading MRREL...


MRREL: 0it [00:00, ?it/s]

UMLS rel entries processed : 2,925,802
Unique directed edges      : 1,557,391
Edges with multiple rels   : 49,604


18

**Cell 7 — Connect corpus chunks**

In [10]:
# ──Connect corpus chunks (5-method + unbiased propagation) ─────────

# Safety check — norm_to_cui must be populated from Cell 5 pass 2
assert norm_to_cui, "norm_to_cui is empty — rerun Cell 5 (MRCONSO pass 2) first"

print('Connecting corpus chunks to graph nodes (5 methods)...')

stats = dict(meshid=0, rxnorm=0, preferred=0, synonym=0, normalised=0, missed=0)

for idx, mesh_list in enumerate(tqdm(df['meshes_norm'], desc='Connecting')):
    for entry in mesh_list:
        term   = entry['term']
        meshid = entry['mesh_id']
        cui    = None

        if meshid and meshid in meshid_to_cui:
            cui = meshid_to_cui[meshid];             stats['meshid'] += 1
        elif meshid and meshid in rxnorm_to_cui:
            cui = rxnorm_to_cui[meshid];             stats['rxnorm'] += 1
        elif term and term in label_to_cui:
            cui = label_to_cui[term];                stats['preferred'] += 1
        elif term and term in synonym_to_cui:
            cui = synonym_to_cui[term];              stats['synonym'] += 1
        else:
            n = _normalise(term) if term else ''
            if n and n in label_to_cui:
                cui = label_to_cui[n];               stats['normalised'] += 1
            elif n and n in norm_to_cui:
                cui = norm_to_cui[n];                stats['normalised'] += 1
            else:
                stats['missed'] += 1

        if cui and cui in G:
            G.nodes[cui]['chunk_idxs'].append(idx)

# term_to_chunks fallback
print('\nApplying term_to_chunks fallback...')
fallback_count = 0
for term, chunk_idxs in tqdm(term_to_chunks.items(), desc='Term fallback'):
    cui = (label_to_cui.get(term) or
           synonym_to_cui.get(term) or
           norm_to_cui.get(_normalise(term)))
    if not cui or cui not in G:
        continue
    existing = set(G.nodes[cui]['chunk_idxs'])
    new_idxs = [i for i in chunk_idxs if i not in existing]
    if new_idxs:
        G.nodes[cui]['chunk_idxs'].extend(new_idxs)
        fallback_count += len(new_idxs)

# Deduplicate
for node in G.nodes():
    G.nodes[node]['chunk_idxs'] = list(set(G.nodes[node]['chunk_idxs']))

reachable_direct = sum(1 for n in G.nodes() if G.nodes[n]['chunk_idxs'])
gc.collect()

# ── Hub node cap ──────────────────────────────────────────────────────────────
import random
HUB_CAP = 50_000
hub_nodes_capped = 0

random.seed(42)  # FIX: seed ONCE outside the loop, not inside it
for node in G.nodes():
    chunks = G.nodes[node]['chunk_idxs']
    if len(chunks) > HUB_CAP:
        G.nodes[node]['chunk_idxs'] = random.sample(chunks, HUB_CAP)
        hub_nodes_capped += 1

print(f'Hub nodes capped at {HUB_CAP:,} chunks : {hub_nodes_capped:,} nodes')
gc.collect()

# ── Neighbour propagation — unbiased round-robin ──────────────────────────────
print('\nRunning neighbour propagation (unbiased round-robin)...')

MAX_INHERIT = 500
propagated  = 0

for node in tqdm(list(G.nodes()), desc='Propagating'):
    if G.nodes[node]['chunk_idxs']:
        continue

    all_nbr_chunks = []
    for nbr in list(G.predecessors(node)) + list(G.successors(node)):
        c = G.nodes[nbr]['chunk_idxs']
        if c:
            all_nbr_chunks.append(c)

    if not all_nbr_chunks:
        continue

    inherited = []
    seen      = set()
    positions = [0] * len(all_nbr_chunks)

    while len(inherited) < MAX_INHERIT:
        made_progress = False
        for i, nbr_chunks in enumerate(all_nbr_chunks):
            while positions[i] < len(nbr_chunks):
                c = nbr_chunks[positions[i]]
                positions[i] += 1
                if c not in seen:
                    seen.add(c)
                    inherited.append(c)
                    made_progress = True
                    break
            if len(inherited) >= MAX_INHERIT:
                break
        if not made_progress:
            break

    if inherited:
        G.nodes[node]['chunk_idxs'] = inherited
        propagated += 1

reachable_after = sum(1 for n in G.nodes() if G.nodes[n]['chunk_idxs'])

print(f'\n── Linking stats ──────────────────────────────────────────')
print(f'  Method 1 mesh_id    : {stats["meshid"]:>10,}')
print(f'  Method 2 rxnorm     : {stats["rxnorm"]:>10,}')
print(f'  Method 3 preferred  : {stats["preferred"]:>10,}')
print(f'  Method 4 synonym    : {stats["synonym"]:>10,}')
print(f'  Method 5 normalised : {stats["normalised"]:>10,}')
print(f'  Term fallback       : {fallback_count:>10,}')
print(f'  Still missed        : {stats["missed"]:>10,}')
print(f'\n  Directly linked     : {reachable_direct:,}')
print(f'  After propagation   : {reachable_after:,} / {G.number_of_nodes():,}'
      f'  ({100*reachable_after/G.number_of_nodes():.1f}%)')
gc.collect()

Connecting corpus chunks to graph nodes (5 methods)...


Connecting:   0%|          | 0/2089296 [00:00<?, ?it/s]


Applying term_to_chunks fallback...


Term fallback:   0%|          | 0/26190 [00:00<?, ?it/s]

Hub nodes capped at 50,000 chunks : 36 nodes

Running neighbour propagation (unbiased round-robin)...


Propagating:   0%|          | 0/1294438 [00:00<?, ?it/s]


── Linking stats ──────────────────────────────────────────
  Method 1 mesh_id    : 22,564,885
  Method 2 rxnorm     :          0
  Method 3 preferred  :     94,084
  Method 4 synonym    :        507
  Method 5 normalised :      1,831
  Term fallback       : 11,179,161
  Still missed        :    446,906

  Directly linked     : 34,183
  After propagation   : 546,549 / 1,294,438  (42.2%)


20

In [11]:
test_terms = ['diabetes mellitus', 'metformin', 'hypertension',
              'female', 'aged', 'rats', 'treatment outcome',
              'breast neoplasms', 'insulin', 'inflammation']

print('Traversal check:')
for term in test_terms:
    cui = label_to_cui.get(term)
    if not cui:
        print(f'  {term:30s} → NOT FOUND')
        continue
    chunks    = G.nodes[cui].get('chunk_idxs', [])
    neighbors = [G.nodes[n].get('label','?')
                 for n in list(G.successors(cui))[:3]]
    print(f'  {term:30s} → {len(chunks):,} chunks | {neighbors}')

Traversal check:
  diabetes mellitus              → 9,262 chunks | ['aspergillus fumigates', 'disorder of endocrine pancreas (disorder)', '[d]hyperglycaemia (situation)']
  metformin                      → 1,278 chunks | ['n,n-dimethylimidodicarbonimidic diamide monohydrochloride', 'sitagliptin phosphate-metformin hydrochloride drug combination', 'metformin hydrochloride 1g tablet']
  hypertension                   → 30,590 chunks | ['hypertension nos (& [essential]) (disorder)', 'acebutolol (substance)', '(hypertensive disease) or (hypertension) (disorder)']
  female                         → 50,000 chunks | ['female gender', 'woman (person)']
  aged                           → 50,000 chunks | ['mestranol (substance)', 'personal status nos (observable entity)', 'aged 80']
  rats                           → 50,000 chunks | ['genus mus', 'old world rat (organism)', 'strains, inbred rat']
  treatment outcome              → 37,911 chunks | ['outcome of event', 'treatment outcome']
  breas

**Cell 8 — Co-occurrence edges**

In [12]:
#  ── Co-occurrence edges (wider lookup cascade) ─────────────────────

# Safety check
assert norm_to_cui, "norm_to_cui is empty — rerun Cell 5 (MRCONSO pass 2) first"

print('Building co-occurrence edges...')

cooccur = defaultdict(int)

for idx, mesh_list in enumerate(tqdm(df['meshes_norm'], desc='Co-occurrence')):
    seen_cuis = set()
    for e in mesh_list:
        meshid = e['mesh_id']
        term   = e['term']
        cui = (
            meshid_to_cui.get(meshid) or
            rxnorm_to_cui.get(meshid) or
            label_to_cui.get(term) or
            synonym_to_cui.get(term) or
            norm_to_cui.get(_normalise(term) if term else '')
        )
        if cui and cui in G:
            seen_cuis.add(cui)

    for a, b in combinations(seen_cuis, 2):
        cooccur[tuple(sorted([a, b]))] += 1

co_edges = 0
for (a, b), w in cooccur.items():
    if w >= MIN_COOCCUR:
        G.add_edge(a, b, rel='CO_OCCUR', weight=w)
        G.add_edge(b, a, rel='CO_OCCUR', weight=w)
        co_edges += 1

print(f'Co-occurrence edges added : {co_edges:,}')
print(f'Total edges now           : {G.number_of_edges():,}')
del cooccur
gc.collect()

Building co-occurrence edges...


Co-occurrence:   0%|          | 0/2089296 [00:00<?, ?it/s]

Co-occurrence edges added : 2,520,531
Total edges now           : 6,588,080


20

**Cell 9 — Save**

In [13]:
# ── Save HKG — single file output ────────────────────────────────────────────
#
# ONE pkl file (hkg.pkl) — Notebook B loads this.
# corpus_texts is NOT saved here — in Notebook B, reload the parquet directly:
#     df = pd.read_parquet(CORPUS_PATH)
#     corpus_texts = df['text'].tolist()
# Row order is guaranteed identical.
#
# CSV files (2) — human-readable inspection only, not used by Notebook B:
#   hkg_nodes.csv : every node with label, TUI, chunk count
#   hkg_edges.csv : every edge with rel type and weight (may be large)

import pickle, csv
from pathlib import Path

OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

# ── Step 1: Build plain-dict graph structures ─────────────────────────────────
print('Converting nx.DiGraph to plain dicts...')

adj          = {}
chunk_index  = {}
cui_to_label = {}

for node in tqdm(G.nodes(), desc='Building structures'):
    data  = G.nodes[node]
    label = data.get('label', '')
    cui_to_label[node] = label

    chunks = data.get('chunk_idxs', [])
    if chunks:
        chunk_index[node] = chunks

    nbrs = []
    for nbr, edata in G[node].items():
        nbrs.append((
            nbr,
            edata.get('rel', ''),
            edata.get('rela', ''),
            float(edata.get('weight', 1)),
        ))
    if nbrs:
        adj[node] = nbrs

print(f'  adj nodes         : {len(adj):,}')
print(f'  chunk_index nodes : {len(chunk_index):,}')

# ── Step 2: Save single hkg.pkl ───────────────────────────────────────────────
hkg_path = OUT_DIR / 'hkg.pkl'
print(f'\nSaving {hkg_path.name}...')
with open(hkg_path, 'wb') as f:
    pickle.dump({
        # graph
        'adj'             : adj,
        'chunk_index'     : chunk_index,
        'cui_to_tui'      : cui_to_tui,
        'cui_to_label'    : cui_to_label,
        # lookups
        'label_to_cui'    : label_to_cui,
        'meshid_to_cui'   : meshid_to_cui,
        'synonym_to_cui'  : synonym_to_cui,
        'rxnorm_to_cui'   : rxnorm_to_cui,
        'norm_to_cui'     : norm_to_cui,
        'term_to_chunks'  : dict(term_to_chunks),
        'meshid_to_chunks': dict(meshid_to_chunks),
    }, f, protocol=4)
print(f'  {hkg_path.stat().st_size/1024**2:.0f} MB')

# ── Step 3: Save hkg_nodes.csv ────────────────────────────────────────────────
tui_name_map = {
    'T047':'Disease','T121':'Drug','T023':'Anatomy','T039':'Physiology',
    'T061':'Procedure','T016':'Human','T008':'Animal','T028':'Gene',
    'T033':'Finding','T192':'Receptor','T082':'SpatialConcept',
    'T102':'GroupAttribute','T091':'BiomedOccupation','T196':'Isotope',
}

nodes_path = OUT_DIR / 'hkg_nodes.csv'
print(f'\nSaving {nodes_path.name}...')
with open(nodes_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['cui','label','tui','tui_name','sab',
                     'n_chunks','has_direct_chunks','n_neighbours'])
    for node in tqdm(G.nodes(), desc='Writing nodes CSV'):
        data         = G.nodes[node]
        tui          = cui_to_tui.get(node, '')
        n_chunks     = len(data.get('chunk_idxs', []))
        has_direct   = node in chunk_index and n_chunks > 0
        n_neighbours = len(list(G.successors(node)))
        writer.writerow([
            node,
            data.get('label',''),
            tui,
            tui_name_map.get(tui, tui),
            data.get('sab',''),
            n_chunks,
            has_direct,
            n_neighbours,
        ])
print(f'  {nodes_path.stat().st_size/1024**2:.1f} MB')

# ── Step 4: Save hkg_edges.csv ────────────────────────────────────────────────
edges_path = OUT_DIR / 'hkg_edges.csv'
print(f'Saving {edges_path.name}...')
with open(edges_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['src_cui','src_label','dst_cui','dst_label',
                     'rel','rela','weight'])
    for u, v, edata in tqdm(G.edges(data=True), desc='Writing edges CSV',
                             total=G.number_of_edges()):
        writer.writerow([
            u, cui_to_label.get(u,''),
            v, cui_to_label.get(v,''),
            edata.get('rel',''),
            edata.get('rela',''),
            edata.get('weight', 1),
        ])
print(f'  {edges_path.stat().st_size/1024**2:.1f} MB')

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'\n{"="*55}')
print(f'SAVE COMPLETE')
print(f'{"="*55}')
print(f'  Nodes              : {G.number_of_nodes():,}')
print(f'  Edges              : {G.number_of_edges():,}')
print(f'  Chunk-linked nodes : {len(chunk_index):,}')
print(f'  Output file        : hkg.pkl')
print(f'  Size               : {hkg_path.stat().st_size/1024**2:.0f} MB')
print(f'\n  In Notebook B, load with:')
print(f'    import pickle')
print(f'    with open("hkg.pkl","rb") as f: hkg = pickle.load(f)')
print(f'    adj            = hkg["adj"]')
print(f'    chunk_index    = hkg["chunk_index"]')
print(f'    cui_to_tui     = hkg["cui_to_tui"]')
print(f'    cui_to_label   = hkg["cui_to_label"]')
print(f'    label_to_cui   = hkg["label_to_cui"]')
print(f'    meshid_to_cui  = hkg["meshid_to_cui"]')
print(f'    synonym_to_cui = hkg["synonym_to_cui"]')
print(f'    rxnorm_to_cui  = hkg["rxnorm_to_cui"]')
print(f'    norm_to_cui    = hkg["norm_to_cui"]')
print(f'\n  corpus_texts: reload from parquet in Notebook B:')
print(f'    df = pd.read_parquet(CORPUS_PATH)')
print(f'    corpus_texts = df["text"].tolist()')

Converting nx.DiGraph to plain dicts...


Building structures:   0%|          | 0/1294438 [00:00<?, ?it/s]

  adj nodes         : 799,983
  chunk_index nodes : 546,549

Saving hkg.pkl...
  1973 MB

Saving hkg_nodes.csv...


Writing nodes CSV:   0%|          | 0/1294438 [00:00<?, ?it/s]

  96.1 MB
Saving hkg_edges.csv...


Writing edges CSV:   0%|          | 0/6588080 [00:00<?, ?it/s]

  550.0 MB

SAVE COMPLETE
  Nodes              : 1,294,438
  Edges              : 6,588,080
  Chunk-linked nodes : 546,549
  Output file        : hkg.pkl
  Size               : 1973 MB

  In Notebook B, load with:
    import pickle
    with open("hkg.pkl","rb") as f: hkg = pickle.load(f)
    adj            = hkg["adj"]
    chunk_index    = hkg["chunk_index"]
    cui_to_tui     = hkg["cui_to_tui"]
    cui_to_label   = hkg["cui_to_label"]
    label_to_cui   = hkg["label_to_cui"]
    meshid_to_cui  = hkg["meshid_to_cui"]
    synonym_to_cui = hkg["synonym_to_cui"]
    rxnorm_to_cui  = hkg["rxnorm_to_cui"]
    norm_to_cui    = hkg["norm_to_cui"]

  corpus_texts: reload from parquet in Notebook B:
    df = pd.read_parquet(CORPUS_PATH)
    corpus_texts = df["text"].tolist()


In [ ]:
# ── Cell 10 — Full Inspection: Stats + Traversal + Visualizations ─────────────

import random

print('=' * 65)
print('SECTION 1 — FINAL KG SUMMARY STATS')
print('=' * 65)

total_nodes     = G.number_of_nodes()
total_edges     = G.number_of_edges()
nodes_with_data = sum(1 for n in G.nodes() if G.nodes[n]['chunk_idxs'])
umls_edges      = sum(1 for u,v,d in G.edges(data=True) if d.get('rel') != 'CO_OCCUR')
cooccur_edges   = sum(1 for u,v,d in G.edges(data=True) if d.get('rel') == 'CO_OCCUR')
avg_degree      = total_edges / max(total_nodes, 1)
isolated        = sum(1 for n in G.nodes() if G.degree(n) == 0)

print(f'  Total nodes              : {total_nodes:,}')
print(f'  Nodes linked to corpus   : {nodes_with_data:,}  ({100*nodes_with_data/total_nodes:.1f}%)')
print(f'  Isolated nodes (degree=0): {isolated:,}')
print(f'  Total edges              : {total_edges:,}')
print(f'  UMLS ontology edges      : {umls_edges:,}')
print(f'  Co-occurrence edges      : {cooccur_edges:,}')
print(f'  Avg degree per node      : {avg_degree:.1f}')

# Lookup table sizes
print(f'\n  Lookup table sizes:')
print(f'    label_to_cui      : {len(label_to_cui):,}')
print(f'    meshid_to_cui     : {len(meshid_to_cui):,}')
print(f'    synonym_to_cui    : {len(synonym_to_cui):,}')
print(f'    rxnorm_to_cui     : {len(rxnorm_to_cui):,}')
print(f'    norm_to_cui       : {len(norm_to_cui):,}')

# SAB breakdown
print(f'\n  Node count by source vocabulary:')
sab_counts = defaultdict(int)
for n in G.nodes():
    sab_counts[G.nodes[n].get('sab', '?')] += 1
for sab, cnt in sorted(sab_counts.items(), key=lambda x: -x[1]):
    print(f'    {sab:15s} : {cnt:,}')

# ── SECTION 2: Traversal test ──────────────────────────────────────────────────
print('\n' + '=' * 65)
print('SECTION 2 — TRAVERSAL TEST (known medical terms)')
print('=' * 65)
print(f'  {"Term":<30s} {"CUI":<12} {"TUI":<6} {"Chunks":>8}  Top 3 neighbors')
print(f'  {"-"*30} {"-"*11} {"-"*5} {"-"*8}  {"-"*30}')

test_terms = [
    'diabetes mellitus', 'metformin', 'hypertension',
    'breast neoplasms', 'insulin', 'myocardial infarction',
    'female', 'aged', 'rats', 'treatment outcome',
    'inflammation', 'obesity', 'lung neoplasms', 'aspirin',
]

all_found = True
for term in test_terms:
    cui = label_to_cui.get(term)
    if not cui:
        print(f'  {term:<30s} NOT IN GRAPH ✗')
        all_found = False
        continue
    chunks    = G.nodes[cui].get('chunk_idxs', [])
    neighbors = list(G.successors(cui))
    nb_labels = [G.nodes[n].get('label', '?')[:20] for n in neighbors[:3]]
    tui       = cui_to_tui.get(cui, '?')
    found_mark = '✓' if chunks else '⚠ no chunks'
    print(f'  {term:<30s} {cui:<12} {tui:<6} {len(chunks):>8,}  {nb_labels}  {found_mark}')

print(f'\n  All core terms found: {"YES ✓" if all_found else "NO — check above ✗"}')

# ── SECTION 3: Sample node table ──────────────────────────────────────────────
print('\n' + '=' * 65)
print('SECTION 3 — SAMPLE NODES (10 random chunk-linked nodes)')
print('=' * 65)

chunk_nodes = [n for n in G.nodes() if G.nodes[n]['chunk_idxs']]
random.seed(0)
sample_nodes = random.sample(chunk_nodes, min(10, len(chunk_nodes)))

rows = []
for node in sample_nodes:
    data    = G.nodes[node]
    succs   = list(G.successors(node))
    nb_labs = [G.nodes[n].get('label','?')[:18] for n in succs[:3]]
    rows.append({
        'CUI'      : node,
        'Label'    : data.get('label','')[:28],
        'SAB'      : data.get('sab',''),
        'TUI'      : cui_to_tui.get(node,'?'),
        'Chunks'   : len(data.get('chunk_idxs',[])),
        'Degree'   : G.degree(node),
        'Neighbors': ' | '.join(nb_labs),
    })

df_sample = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 35)
pd.set_option('display.width', 130)
print(df_sample.to_string(index=False))

# ── SECTION 4: Output file verification ───────────────────────────────────────
print('\n' + '=' * 65)
print('SECTION 4 — OUTPUT FILE VERIFICATION')
print('=' * 65)

OUT_DIR = Path('/kaggle/working')
output_files = ['hkg.pkl', 'hkg_nodes.csv', 'hkg_edges.csv']
all_ok = True
for fname in output_files:
    fpath = OUT_DIR / fname
    if fpath.exists():
        print(f'  {fname:25s} : {fpath.stat().st_size/1024**2:.1f} MB  ✓')
    else:
        print(f'  {fname:25s} : NOT FOUND  ✗')
        all_ok = False

# Reload hkg.pkl and confirm keys
print('\n  Verifying hkg.pkl keys...')
with open(OUT_DIR / 'hkg.pkl', 'rb') as f:
    hkg_check = pickle.load(f)

expected_keys = [
    'adj','chunk_index','cui_to_tui','cui_to_label',
    'label_to_cui','meshid_to_cui','synonym_to_cui',
    'rxnorm_to_cui','norm_to_cui','term_to_chunks','meshid_to_chunks'
]
for k in expected_keys:
    present = k in hkg_check
    size    = len(hkg_check[k]) if present else 0
    mark    = '✓' if present else '✗'
    print(f'    {k:<22s} : {size:>10,} entries  {mark}')

del hkg_check
gc.collect()

print(f'\n  Files OK: {"YES ✓" if all_ok else "SOME MISSING ✗"}')

# ── SECTION 5: Visualizations ─────────────────────────────────────────────────
print('\n' + '=' * 65)
print('SECTION 5 — VISUALIZATIONS')
print('=' * 65)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Hybrid Knowledge Graph — Build Summary', fontsize=16, fontweight='bold')

# Plot 1: Nodes by semantic type
ax1 = axes[0, 0]
tui_name_map = {
    'T047':'Disease','T121':'Drug','T023':'Anatomy',
    'T039':'Physiology','T061':'Procedure','T016':'Human',
    'T008':'Animal','T028':'Gene','T033':'Finding',
    'T062':'Research','T168':'Food','T053':'Behavior',
    'T192':'Receptor','T196':'Isotope','T082':'SpatialConcept',
}
tui_counts = defaultdict(int)
for node in G.nodes():
    tui = cui_to_tui.get(node, 'Other')
    tui_counts[tui_name_map.get(tui, tui)] += 1
top_tuis = sorted(tui_counts.items(), key=lambda x: x[1], reverse=True)[:14]
labels_t, counts_t = zip(*top_tuis)
colors1 = plt.cm.Set3(np.linspace(0, 1, len(labels_t)))
bars1 = ax1.barh(labels_t, counts_t, color=colors1)
ax1.set_xlabel('Node count')
ax1.set_title('Nodes by Semantic Type')
for bar, count in zip(bars1, counts_t):
    ax1.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
             f'{count:,}', va='center', fontsize=7)
ax1.invert_yaxis()

# Plot 2: Edge type distribution
ax2 = axes[0, 1]
edge_type_counts = defaultdict(int)
for u, v, data in G.edges(data=True):
    edge_type_counts[data.get('rel', 'UNK')] += 1
et_labels = list(edge_type_counts.keys())
et_values = list(edge_type_counts.values())
colors2 = plt.cm.Pastel1(np.linspace(0, 1, len(et_labels)))
ax2.pie(et_values, labels=et_labels, autopct='%1.1f%%',
        colors=colors2, startangle=90)
ax2.set_title('Edge Type Distribution')

# Plot 3: Chunks per node (log scale)
ax3 = axes[0, 2]
chunk_counts = [len(G.nodes[n]['chunk_idxs'])
                for n in G.nodes() if G.nodes[n]['chunk_idxs']]
ax3.hist(chunk_counts, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
ax3.set_xlabel('Chunks per node')
ax3.set_ylabel('Number of nodes')
ax3.set_title('Chunk Coverage Distribution (log scale)')
ax3.set_yscale('log')
median_chunks = np.median(chunk_counts)
ax3.axvline(median_chunks, color='red', linestyle='--',
            label=f'Median: {median_chunks:.0f}')
ax3.axvline(np.percentile(chunk_counts, 95), color='orange', linestyle=':',
            label=f'P95: {np.percentile(chunk_counts,95):.0f}')
ax3.legend(fontsize=8)

# Plot 4: Nodes by SAB
ax4 = axes[1, 0]
sab_labels = list(sab_counts.keys())
sab_values = list(sab_counts.values())
colors4 = plt.cm.Accent(np.linspace(0, 1, len(sab_labels)))
bars4 = ax4.bar(sab_labels, sab_values, color=colors4, alpha=0.85, edgecolor='white')
ax4.set_ylabel('Node count')
ax4.set_title('Nodes by Source Vocabulary')
for bar, count in zip(bars4, sab_values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{count:,}', ha='center', fontsize=8)

# Plot 5: MeSH coverage by corpus source
ax5 = axes[1, 1]
source_names  = [s for s in ['pqaa','pqau','medrag_pubmed','medrag_wikipedia']
                 if s in df['source'].values]
source_colors = ['#4CAF50','#2196F3','#FF9800','#9C27B0'][:len(source_names)]
source_counts = []
for src in source_names:
    sub = df[df['source'] == src]
    has_mesh = sub['meshes_norm'].apply(lambda x: len(x) > 0).sum()
    source_counts.append(has_mesh)
bars5 = ax5.bar(source_names, source_counts,
                color=source_colors, alpha=0.85, edgecolor='white')
ax5.set_ylabel('Chunks with MeSH')
ax5.set_title('MeSH Coverage by Corpus Source')
ax5.tick_params(axis='x', rotation=15)
for bar, count in zip(bars5, source_counts):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{count:,}', ha='center', fontsize=8)

# Plot 6: Degree distribution (top 20 highest-degree nodes)
ax6 = axes[1, 2]
degree_seq = sorted([(G.degree(n), G.nodes[n].get('label','')[:20])
                     for n in G.nodes()], reverse=True)[:20]
deg_vals, deg_labels = zip(*degree_seq)
colors6 = plt.cm.YlOrRd(np.linspace(0.3, 1.0, len(deg_vals)))
bars6 = ax6.barh(deg_labels, deg_vals, color=colors6)
ax6.set_xlabel('Degree')
ax6.set_title('Top 20 Highest-Degree Nodes')
ax6.invert_yaxis()
for bar, val in zip(bars6, deg_vals):
    ax6.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig('/kaggle/working/hkg_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hkg_summary.png')
print('\nInspection complete.')

SECTION 1 — FINAL KG SUMMARY STATS
  Total nodes              : 1,294,438
  Nodes linked to corpus   : 546,549  (42.2%)
  Isolated nodes (degree=0): 487,206
  Total edges              : 6,588,080
  UMLS ontology edges      : 1,547,018
  Co-occurrence edges      : 5,041,062
  Avg degree per node      : 5.1

  Lookup table sizes:
    label_to_cui      : 3,275,438
    meshid_to_cui     : 29,121
    synonym_to_cui    : 3,324,271
    rxnorm_to_cui     : 224,477
    norm_to_cui       : 45,318

  Node count by source vocabulary:
    MSH             : 432,173
    SNOMEDCT_US     : 389,964
    RXNORM          : 218,030
    NCI             : 158,599
    ICD10CM         : 95,672

SECTION 2 — TRAVERSAL TEST (known medical terms)
  Term                           CUI          TUI      Chunks  Top 3 neighbors
  ------------------------------ ----------- ----- --------  ------------------------------
  diabetes mellitus              C0011849     T047      9,262  ['aspergillus fumigate', 'disorder of e